In [4]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import json
from features import load_clean_data, get_src_cols, compute_risk 
from scenario_engine004 import run_product_scenario

cat_cols = ['product_name', 'category', 'unit']
import inspect
from scenario_engine004 import run_product_scenario
# print(inspect.getsource(run_product_scenario))



In [9]:
pd.set_option('display.max_rows', None)        # show all rows
pd.set_option('display.max_columns', None)     # show all columns
# pd.set_option('display.expand_frame_repr', False)  # don't wrap with ...
# pd.set_option('display.max_colwidth', None)    # show full column content

# ---- load everything once ----
d = load_clean_data()
src_cols = get_src_cols(d)
imp_cols = [c for c in ['india', 'china', 'bhutan'] if c in src_cols]

feature_cols = (
    ['product_name', 'category', 'unit', 'm_sin', 'm_cos', 'n_sources',
     'herfindahl', 'import_share', 'domestic_share', 'n_months_present',
     'india_share', 'china_share', 'bhutan_share', 'avg_price_lag1']
    + src_cols
)

booster = lgb.Booster(model_file='../models/price_surrogate_final.txt')
product_vol_stats = pd.read_parquet('../data/processed/product_volume_stats.parquet').reset_index()
with open('../data/processed/risk_scaling_ref.json') as f:
    scaling_ref = json.load(f)

# ---- test a mix: heavy India sourcing, sparse India sourcing, zero India sourcing ----
test_products = ['Orange_Sweet', 'Apple', 'Pear', 'Potato', 'Sugarcane']

print(f"{'Product':<15} {'Status':<20} {'Coverage':<30} {'BaselinePrice':>13} {'ShockedPrice':>13} {'Delta%':>8} {'Risk':>8}")
print("-" * 115)

rows = []
for p in test_products:
    result = run_product_scenario(
        d, p, {'india': 0.7}, booster, feature_cols,
        cat_cols, product_vol_stats, scaling_ref, src_cols, imp_cols
    )
    rows.append(result)

results_df = pd.DataFrame(rows)
# results_df.head()
    

Product         Status               Coverage                       BaselinePrice  ShockedPrice   Delta%     Risk
-------------------------------------------------------------------------------------------------------------------


## INDIA -30% supply (Hypothetical)
All products results in df (Externally not Saved)

In [8]:
# India -30%, all products
from scenario_runner import run_scenario_report


results_india, summary_india = run_scenario_report(
    d, {'india': 0.7}, products=None,
    price_booster=booster, feature_cols=feature_cols, cat_cols=cat_cols,
    product_vol_stats=product_vol_stats, scaling_ref=scaling_ref,
    src_cols=src_cols, imp_cols=imp_cols
)
results_india.head()

,product_name,status,baseline_month,coverage,confidence,baseline_price,predicted_price,price_delta_pct,predicted_volume,baseline_risk,baseline_risk_score,risk_score,risk
0,Akabary_Chilly,ok,month 10,india: sourced in 2/5 months,india: low-confidence,450.0,432.313882,-3.930248,13030.500000,Low,0.278951,0.268854,Low
1,Amala,ok,month 10,india: sourced in 8/9 months,india: reliable,145.0,127.833366,-11.839058,11166.312500,Low,0.063284,0.547463,Medium
2,Apple,ok,month 10,india: sourced in 6/10 months,india: reliable,266.3,255.386840,-4.098070,413769.316667,Medium,0.580614,0.445999,Medium
3,Apple_Fuji,no_applicable_shock,month 10,india: never sourced (7 months present),,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arum,ok,month 10,india: sourced in 5/10 months,india: reliable,65.0,71.799111,10.460170,4483.560000,Low,0.312763,0.420477,Medium


## Local +20% 

In [11]:
# Local +20%, all products
results_local, summary_local = run_scenario_report(
    d, {'local': 1.2}, products=None,
    price_booster=booster, feature_cols=feature_cols, cat_cols=cat_cols,
    product_vol_stats=product_vol_stats, scaling_ref=scaling_ref,
    src_cols=src_cols, imp_cols=imp_cols
)
results_local.tail()


,product_name,status,baseline_month,coverage,confidence,baseline_price,predicted_price,price_delta_pct,predicted_volume,baseline_risk,baseline_risk_score,risk_score,risk
90,Tomato_Small,ok,month 10,local: sourced in 9/10 months,local: reliable,69.39,63.605663,-8.335980,7.242009e+05,Low,0.224144,0.220514,Low
91,Turnip,ok,month 10,local: sourced in 4/9 months,local: reliable,110.00,110.434596,0.395087,2.163000e+02,Low,0.142738,0.357054,Medium
92,Walnut,no_applicable_shock,"month 3 (fallback, no month-10 data)",local: never sourced (1 months present),,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93,Watermelon,ok,month 10,local: sourced in 1/10 months,local: single-observation,41.63,46.316152,11.256671,1.757571e+06,Medium,0.377237,0.377216,Medium
94,Yam,ok,month 10,local: sourced in 8/8 months,local: reliable,110.00,96.736268,-12.057938,2.507625e+04,Medium,0.370514,0.370514,Medium


In [12]:
print(summary_india)

{'n_products_affected': 67, 'n_products_na': np.int64(28), 'mean_price_delta_pct': np.float64(7.3937077015306), 'median_price_delta_pct': np.float64(-0.21752699344603776), 'n_products_risk_increased': np.int64(7)}


In [13]:
print(results_india[results_india['status']=='ok'].sort_values('price_delta_pct', ascending=False).head(10))
print(results_india[results_india['status']=='ok'].sort_values('price_delta_pct').head(10))

    product_name status                        baseline_month  \
86  Sweet_Potato     ok                              month 10   
64        Potato     ok  month 6 (fallback, no month-10 data)   
48         Mango     ok                              month 10   
13  Bitter_Gourd     ok                              month 10   
49   Mango_Green     ok                              month 10   
14  Bottle_Gourd     ok                              month 10   
27       Cow_Pea     ok                              month 10   
52         Okara     ok                              month 10   
19       Cabbage     ok                              month 10   
18      Broccoli     ok                              month 10   

                          coverage                 confidence  baseline_price  \
86    india: sourced in 7/8 months            india: reliable           30.00   
64    india: sourced in 1/1 months  india: single-observation           32.84   
48  india: sourced in 10/10 months       